# Advanced Problems with Solutions: `OrderedDict` vs Plain `dict`

This notebook builds on the supplied topic: comparing `collections.OrderedDict` with insertion-ordered plain dictionaries.

## What you will practice

- insertion order and re-insertion behavior
- reverse iteration
- first/last item access and removal
- emulating `OrderedDict.popitem(last=...)`
- emulating `OrderedDict.move_to_end(...)`
- order-sensitive equality
- mutation hazards during iteration
- stable de-duplication
- LRU / recency tracking
- algorithmic complexity
- careful benchmarking with `timeit`
- randomized invariant testing
- designing reusable ordered-mapping utilities

## Source-derived core ideas

The source emphasizes four capabilities traditionally associated with `OrderedDict` beyond ordinary mapping behavior:

1. reverse iteration
2. popping the first or last item
3. moving a key to the beginning or end
4. equality that can take key order into account

It also compares performance of plain `dict` and `OrderedDict`.

> **Modern Python note:** Current Python versions guarantee insertion order for `dict`. Some operations that were awkward or unavailable in older versions are now directly supported. The exercises below preserve the conceptual comparison while using modern, clear Python where appropriate.


## 0. Setup and helper utilities

Run this cell first.


In [1]:
from collections import OrderedDict
from collections.abc import Mapping
from timeit import repeat, timeit
from statistics import mean, median
import random
import sys


In [2]:
def show_mapping(label, mapping):
    """Pretty-print a mapping as an ordered list of pairs."""
    print(f"{label}: {list(mapping.items())}")


def assert_same_items_and_order(left, right):
    """Assert that two mappings have exactly the same ordered item sequence."""
    assert list(left.items()) == list(right.items()), (
        f"Different order/content:\nleft={list(left.items())}\n"
        f"right={list(right.items())}"
    )


# Part I — Order Semantics

## Problem 1 — Updating vs deleting-and-reinserting

Create a dictionary with keys `a, b, c, d`.

1. Update the value of `b`.
2. Verify that `b` does **not** move.
3. Delete `b`, then insert it again.
4. Verify that `b` now appears at the end.

### Why this matters

A recency algorithm often depends on whether an ordinary assignment changes key position. It does not: assigning to an existing key updates its value in place. Delete/reinsert or pop/reinsert changes its insertion position.


In [3]:
# Solution 1

d = {"a": 1, "b": 2, "c": 3, "d": 4}
original_order = list(d)

d["b"] = 200
assert list(d) == original_order
print("After ordinary update:", list(d.items()))

value = d.pop("b")
d["b"] = value

assert list(d) == ["a", "c", "d", "b"]
print("After pop + reinsert:", list(d.items()))


After ordinary update: [('a', 1), ('b', 200), ('c', 3), ('d', 4)]
After pop + reinsert: [('a', 1), ('c', 3), ('d', 4), ('b', 200)]


### Extra example — `OrderedDict` behaves similarly for plain assignment

`OrderedDict.move_to_end()` exists specifically because ordinary assignment does not automatically make an existing key "most recent".


In [4]:
od = OrderedDict(a=1, b=2, c=3, d=4)

od["b"] = 200
assert list(od) == ["a", "b", "c", "d"]

od.move_to_end("b")
assert list(od) == ["a", "c", "d", "b"]

show_mapping("OrderedDict after move_to_end", od)


OrderedDict after move_to_end: [('a', 1), ('c', 3), ('d', 4), ('b', 200)]


## Problem 2 — Reverse iteration without unnecessary copies

Given a mapping, print:

1. keys from newest to oldest,
2. items from newest to oldest,
3. values from newest to oldest.

Then compare a direct reverse-iteration approach with a materialized-list approach.


In [5]:
# Solution 2

d = {"a": 10, "b": 20, "c": 30, "d": 40}

print("Reverse keys:")
for key in reversed(d):
    print(key)

print("\nReverse items:")
for key, value in reversed(d.items()):
    print(key, value)

print("\nReverse values:")
for value in reversed(d.values()):
    print(value)


Reverse keys:
d
c
b
a

Reverse items:
d 40
c 30
b 20
a 10

Reverse values:
40
30
20
10


In [6]:
# A list-based alternative creates an additional list object.
reverse_keys_via_list = list(reversed(list(d.keys())))
reverse_keys_direct = list(reversed(d))

assert reverse_keys_via_list == reverse_keys_direct
print(reverse_keys_direct)


['d', 'c', 'b', 'a']


## Problem 3 — Peek at the first and last item without modifying the mapping

Implement:

```python
first_item(d)
last_item(d)
```

Requirements:

- return `(key, value)`,
- do not mutate `d`,
- raise `KeyError` for an empty mapping,
- avoid converting the entire dictionary to a list.


In [7]:
# Solution 3

def first_item(d):
    if not d:
        raise KeyError("mapping is empty")
    key = next(iter(d))
    return key, d[key]


def last_item(d):
    if not d:
        raise KeyError("mapping is empty")
    key = next(reversed(d))
    return key, d[key]


d = {"a": 1, "b": 2, "c": 3}

assert first_item(d) == ("a", 1)
assert last_item(d) == ("c", 3)
assert list(d) == ["a", "b", "c"]

print("first:", first_item(d))
print("last :", last_item(d))


first: ('a', 1)
last : ('c', 3)


### Edge-case test


In [8]:
for fn in (first_item, last_item):
    try:
        fn({})
    except KeyError as exc:
        print(fn.__name__, "->", repr(exc))
    else:
        raise AssertionError("Expected KeyError")


first_item -> KeyError('mapping is empty')
last_item -> KeyError('mapping is empty')


# Part II — Pop Operations

## Problem 4 — Recreate `OrderedDict.popitem(last=...)` for a plain `dict`

Write:

```python
popitem_plain(d, *, last=True)
```

Behavior:

- `last=True`: remove and return the last item.
- `last=False`: remove and return the first item.
- empty mapping: raise `KeyError`.

Avoid building a full list of keys.


In [9]:
# Solution 4

def popitem_plain(d, *, last=True):
    if not d:
        raise KeyError("dictionary is empty")

    if last:
        return d.popitem()

    first_key = next(iter(d))
    return first_key, d.pop(first_key)


d = {"a": 1, "b": 2, "c": 3, "d": 4}

assert popitem_plain(d, last=False) == ("a", 1)
assert list(d) == ["b", "c", "d"]

assert popitem_plain(d, last=True) == ("d", 4)
assert list(d) == ["b", "c"]

print(d)


{'b': 2, 'c': 3}


## Problem 5 — Drain from both ends

Starting with keys `0..9`, alternately pop:

- the first item,
- then the last item,
- then the first item,
- then the last item,
- and so on.

Return the pop sequence.


In [10]:
# Solution 5

def drain_alternating(d):
    result = []
    pop_first = True

    while d:
        result.append(popitem_plain(d, last=not pop_first))
        pop_first = not pop_first

    return result


d = {i: i * i for i in range(10)}
sequence = drain_alternating(d)

print(sequence)
assert sequence == [
    (0, 0), (9, 81),
    (1, 1), (8, 64),
    (2, 4), (7, 49),
    (3, 9), (6, 36),
    (4, 16), (5, 25),
]
assert d == {}


[(0, 0), (9, 81), (1, 1), (8, 64), (2, 4), (7, 49), (3, 9), (6, 36), (4, 16), (5, 25)]


## Problem 6 — Generic `pop_ends`

Implement a function that removes up to `n_first` items from the front and `n_last` items from the end.

Return two lists: `(front_removed, back_removed)`.

If the dictionary runs out of items, stop cleanly.


In [11]:
# Solution 6

def pop_ends(d, *, n_first=0, n_last=0):
    if n_first < 0 or n_last < 0:
        raise ValueError("counts must be non-negative")

    front = []
    back = []

    for _ in range(n_first):
        if not d:
            break
        front.append(popitem_plain(d, last=False))

    for _ in range(n_last):
        if not d:
            break
        back.append(popitem_plain(d, last=True))

    return front, back


d = {ch: i for i, ch in enumerate("abcdef", start=1)}
front, back = pop_ends(d, n_first=2, n_last=3)

print("front:", front)
print("back :", back)
print("left :", d)

assert front == [("a", 1), ("b", 2)]
assert back == [("f", 6), ("e", 5), ("d", 4)]
assert list(d.items()) == [("c", 3)]


front: [('a', 1), ('b', 2)]
back : [('f', 6), ('e', 5), ('d', 4)]
left : {'c': 3}


# Part III — Moving Keys

## Problem 7 — Move a key to the end with a plain `dict`

Write an in-place function:

```python
move_to_end_plain(d, key)
```

Match the practical effect of `OrderedDict.move_to_end(key, last=True)`.


In [12]:
# Solution 7

def move_to_end_plain(d, key):
    d[key] = d.pop(key)


d = {"a": 1, "b": 2, "c": 3, "d": 4}
move_to_end_plain(d, "b")

assert list(d) == ["a", "c", "d", "b"]
print(d)


{'a': 1, 'c': 3, 'd': 4, 'b': 2}


## Problem 8 — Move a key to the front with a plain `dict`

Implement:

```python
move_to_front_plain(d, key)
```

Do it **in place** while preserving the relative order of all other keys.

A useful strategy is:

1. move the target to the end,
2. rotate every key that currently precedes it to the end.


In [13]:
# Solution 8 — rotation approach

def move_to_front_plain(d, key):
    d[key] = d.pop(key)
    preceding = list(d)[:-1]

    for k in preceding:
        d[k] = d.pop(k)


d = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
move_to_front_plain(d, "c")

assert list(d) == ["c", "a", "b", "d", "e"]
print(d)


{'c': 3, 'a': 1, 'b': 2, 'd': 4, 'e': 5}


### Alternative solution — rebuild order explicitly

This version is often easier to read. It still has linear work and allocates a new sequence of pairs.


In [14]:
def move_to_front_rebuild(d, key):
    value = d[key]

    reordered = [(key, value)]
    reordered.extend((k, v) for k, v in d.items() if k != key)

    d.clear()
    d.update(reordered)


d = {"a": 1, "b": 2, "c": 3, "d": 4}
move_to_front_rebuild(d, "d")

assert list(d) == ["d", "a", "b", "c"]
print(d)


{'d': 4, 'a': 1, 'b': 2, 'c': 3}


## Problem 9 — One API for both directions

Implement:

```python
move_to_end_compatible(d, key, *, last=True)
```

- If `last=True`, move `key` to the end.
- If `last=False`, move `key` to the beginning.


In [15]:
# Solution 9

def move_to_end_compatible(d, key, *, last=True):
    if last:
        move_to_end_plain(d, key)
    else:
        move_to_front_plain(d, key)


d = {"a": 1, "b": 2, "c": 3, "d": 4}

move_to_end_compatible(d, "b")
assert list(d) == ["a", "c", "d", "b"]

move_to_end_compatible(d, "d", last=False)
assert list(d) == ["d", "a", "c", "b"]

print(d)


{'d': 4, 'a': 1, 'c': 3, 'b': 2}


## Problem 10 — Differential test against `OrderedDict`

Use an `OrderedDict` as a behavioral reference.

Apply the same sequence of move operations to:

- one `OrderedDict`,
- one plain `dict` using your compatibility function.

Assert that their item order stays identical.


In [16]:
# Solution 10

operations = [
    ("c", True),
    ("a", True),
    ("d", False),
    ("b", False),
    ("e", True),
]

plain = dict(a=1, b=2, c=3, d=4, e=5)
ordered = OrderedDict(plain)

for key, last in operations:
    move_to_end_compatible(plain, key, last=last)
    ordered.move_to_end(key, last=last)
    assert_same_items_and_order(plain, ordered)

print("Final plain  :", list(plain.items()))
print("Final ordered:", list(ordered.items()))


Final plain  : [('b', 2), ('d', 4), ('c', 3), ('a', 1), ('e', 5)]
Final ordered: [('b', 2), ('d', 4), ('c', 3), ('a', 1), ('e', 5)]


# Part IV — Equality and Order-Sensitive Comparison

## Problem 11 — Why `dict` equality is not enough

Create two dictionaries containing the same key/value pairs in different insertion orders.

Show that:

- ordinary dictionary equality is `True`,
- ordered item-sequence equality is `False`.


In [17]:
# Solution 11

d1 = {"a": 10, "b": 20, "c": 30}
d2 = {"b": 20, "c": 30, "a": 10}

print("d1 == d2:", d1 == d2)
print(
    "list(d1.items()) == list(d2.items()):",
    list(d1.items()) == list(d2.items())
)

assert d1 == d2
assert list(d1.items()) != list(d2.items())


d1 == d2: True
list(d1.items()) == list(d2.items()): False


## Problem 12 — Implement order-sensitive mapping equality

Implement:

```python
ordered_mapping_equal(left, right)
```

Requirements:

- both objects must be mappings,
- key order must match,
- each corresponding value must compare equal,
- avoid separately materializing full key lists.


In [18]:
# Solution 12

def ordered_mapping_equal(left, right):
    if not isinstance(left, Mapping) or not isinstance(right, Mapping):
        return False

    if len(left) != len(right):
        return False

    return all(
        k1 == k2 and v1 == v2
        for (k1, v1), (k2, v2) in zip(left.items(), right.items())
    )


a = {"x": 1, "y": 2}
b = {"x": 1, "y": 2}
c = {"y": 2, "x": 1}
d = {"x": 1, "y": 999}

assert ordered_mapping_equal(a, b)
assert not ordered_mapping_equal(a, c)
assert not ordered_mapping_equal(a, d)
assert not ordered_mapping_equal(a, [("x", 1), ("y", 2)])

print("All tests passed.")


All tests passed.


## Problem 13 — Configurable equality

Write one function that supports two modes:

- `order_sensitive=False`: mapping equality
- `order_sensitive=True`: ordered pair equality


In [19]:
# Solution 13

def mapping_equal(left, right, *, order_sensitive=False):
    if not isinstance(left, Mapping) or not isinstance(right, Mapping):
        return False

    if not order_sensitive:
        return left == right

    return ordered_mapping_equal(left, right)


d1 = {"a": 1, "b": 2}
d2 = {"b": 2, "a": 1}

assert mapping_equal(d1, d2)
assert not mapping_equal(d1, d2, order_sensitive=True)

print("order-insensitive:", mapping_equal(d1, d2))
print("order-sensitive  :", mapping_equal(d1, d2, order_sensitive=True))


order-insensitive: True
order-sensitive  : False


## Problem 14 — Compare `OrderedDict` with `OrderedDict`

Demonstrate that two `OrderedDict` objects with the same pairs but different order can compare unequal.

Then compare their ordinary `dict` conversions.


In [20]:
# Solution 14

od1 = OrderedDict([("a", 1), ("b", 2), ("c", 3)])
od2 = OrderedDict([("b", 2), ("a", 1), ("c", 3)])

print("OrderedDict equality:", od1 == od2)
print("dict equality       :", dict(od1) == dict(od2))

assert od1 != od2
assert dict(od1) == dict(od2)


OrderedDict equality: False
dict equality       : True


# Part V — Mutation Hazards and Safe Patterns

## Problem 15 — Mutating while iterating

Explain and demonstrate why this pattern is unsafe:

```python
for key in d:
    if should_remove(key):
        d.pop(key)
```

Then show two safe alternatives.


In [21]:
# Solution 15A — unsafe pattern

d = {i: i * 10 for i in range(6)}

try:
    for key in d:
        if key % 2 == 0:
            d.pop(key)
except RuntimeError as exc:
    print("Expected RuntimeError:", exc)


Expected RuntimeError: dictionary changed size during iteration


In [22]:
# Solution 15B — safe: iterate over a snapshot

d = {i: i * 10 for i in range(6)}

for key in list(d):
    if key % 2 == 0:
        d.pop(key)

assert list(d) == [1, 3, 5]
print(d)


{1: 10, 3: 30, 5: 50}


In [23]:
# Solution 15C — safe: construct a filtered dictionary

d = {i: i * 10 for i in range(6)}
filtered = {k: v for k, v in d.items() if k % 2 == 1}

assert list(filtered) == [1, 3, 5]
print(filtered)


{1: 10, 3: 30, 5: 50}


## Problem 16 — Stable de-duplication

Given a sequence with duplicates, return unique values while preserving first-seen order.

Then produce a second version preserving **last-seen** order.


In [24]:
# Solution 16A — preserve first occurrence

def unique_first_seen(values):
    return list(dict.fromkeys(values))


data = ["b", "a", "b", "c", "a", "d", "c"]
result = unique_first_seen(data)

assert result == ["b", "a", "c", "d"]
print(result)


['b', 'a', 'c', 'd']


In [25]:
# Solution 16B — preserve last occurrence order

def unique_last_seen(values):
    d = {}

    for value in values:
        if value in d:
            d.pop(value)
        d[value] = None

    return list(d)


result = unique_last_seen(data)

assert result == ["b", "a", "d", "c"]
print(result)


['b', 'a', 'd', 'c']


# Part VI — Recency Tracking and LRU Patterns

## Problem 17 — Fixed-size recent-items dictionary

Implement a function:

```python
record_recent(d, key, value, *, capacity)
```

Rules:

1. Update or insert the key.
2. Treat the touched key as most recent.
3. If size exceeds `capacity`, remove the least-recent key.
4. Return the evicted `(key, value)` pair, or `None`.

Use a plain `dict`.


In [26]:
# Solution 17

def record_recent(d, key, value, *, capacity):
    if capacity < 1:
        raise ValueError("capacity must be >= 1")

    if key in d:
        d.pop(key)

    d[key] = value

    if len(d) > capacity:
        return popitem_plain(d, last=False)

    return None


recent = {}

assert record_recent(recent, "a", 1, capacity=3) is None
assert record_recent(recent, "b", 2, capacity=3) is None
assert record_recent(recent, "c", 3, capacity=3) is None

assert record_recent(recent, "a", 100, capacity=3) is None
assert list(recent) == ["b", "c", "a"]

evicted = record_recent(recent, "d", 4, capacity=3)

assert evicted == ("b", 2)
assert list(recent.items()) == [("c", 3), ("a", 100), ("d", 4)]

print("evicted:", evicted)
print("recent :", recent)


evicted: ('b', 2)
recent : {'c': 3, 'a': 100, 'd': 4}


## Problem 18 — LRU cache class with `OrderedDict`

Implement a small LRU cache.

Required operations:

- `get(key)` → return value and mark as most recently used
- `put(key, value)` → insert/update and mark as most recently used
- automatic least-recently-used eviction
- `__len__`
- readable `__repr__`


In [27]:
# Solution 18

class LRUCache:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be >= 1")

        self.capacity = capacity
        self._data = OrderedDict()

    def get(self, key):
        value = self._data[key]
        self._data.move_to_end(key)
        return value

    def put(self, key, value):
        evicted = None

        if key in self._data:
            self._data[key] = value
            self._data.move_to_end(key)
        else:
            self._data[key] = value

        if len(self._data) > self.capacity:
            evicted = self._data.popitem(last=False)

        return evicted

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"LRUCache(capacity={self.capacity}, data={list(self._data.items())})"


In [28]:
# Tests for Solution 18

cache = LRUCache(3)

cache.put("a", 1)
cache.put("b", 2)
cache.put("c", 3)

assert cache.get("a") == 1
evicted = cache.put("d", 4)

assert evicted == ("b", 2)
assert list(cache._data.items()) == [
    ("c", 3),
    ("a", 1),
    ("d", 4),
]

print(cache)


LRUCache(capacity=3, data=[('c', 3), ('a', 1), ('d', 4)])


## Problem 19 — LRU cache using only a plain `dict`

Re-implement the previous cache without `OrderedDict`.

This is a useful design exercise: modern `dict` gives insertion order, but you must explicitly express recency updates by pop/reinsert.


In [29]:
# Solution 19

class PlainDictLRUCache:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be >= 1")

        self.capacity = capacity
        self._data = {}

    def get(self, key):
        value = self._data.pop(key)
        self._data[key] = value
        return value

    def put(self, key, value):
        if key in self._data:
            self._data.pop(key)

        self._data[key] = value

        if len(self._data) > self.capacity:
            return popitem_plain(self._data, last=False)

        return None

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return (
            f"PlainDictLRUCache(capacity={self.capacity}, "
            f"data={list(self._data.items())})"
        )


In [30]:
# Differential test: OrderedDict LRU vs plain-dict LRU

ops = [
    ("put", "a", 1),
    ("put", "b", 2),
    ("put", "c", 3),
    ("get", "a", None),
    ("put", "d", 4),
    ("get", "c", None),
    ("put", "e", 5),
]

a = LRUCache(3)
b = PlainDictLRUCache(3)

for op, key, value in ops:
    if op == "put":
        evicted_a = a.put(key, value)
        evicted_b = b.put(key, value)
        assert evicted_a == evicted_b
    else:
        assert a.get(key) == b.get(key)

    assert list(a._data.items()) == list(b._data.items())

print("Final OrderedDict cache:", a)
print("Final plain-dict cache  :", b)


Final OrderedDict cache: LRUCache(capacity=3, data=[('d', 4), ('c', 3), ('e', 5)])
Final plain-dict cache  : PlainDictLRUCache(capacity=3, data=[('d', 4), ('c', 3), ('e', 5)])


# Part VII — Algorithmic Complexity

## Problem 20 — Classify common operations

Fill in the expected high-level complexity under ordinary hash-table assumptions.

| Operation | Plain `dict` | `OrderedDict` |
|---|---:|---:|
| lookup by key | ? | ? |
| insert/update by key | ? | ? |
| pop last item | ? | ? |
| move existing key to end | ? | ? |
| move existing key to front | ? | ? |
| compare full mappings | ? | ? |

### Solution

Typical expected behavior:

- lookup/insert/update: average **O(1)**
- pop last: typically **O(1)**
- `OrderedDict.move_to_end(...)`: designed for efficient reordering, typically **O(1)**
- emulating a move-to-front with the rotation technique on a plain dict: **O(n)**
- full equality comparison: **O(n)** in the ordinary case

Big-O alone is not enough: constants, memory layout, Python version, key types, cache behavior, and the exact workload can strongly affect real timings.


# Part VIII — Benchmarking Best Practices

The supplied material uses `timeit` to compare construction, lookup, and popping. The next problems extend that idea while avoiding common benchmark mistakes.

## Problem 21 — Benchmark construction

Compare construction of:

- plain `dict`
- `OrderedDict`

Use repeated measurements rather than trusting one run.


In [31]:
# Solution 21

def create_dict(n):
    d = {}
    for i in range(n):
        d[i] = i
    return d


def create_ordered_dict(n):
    d = OrderedDict()
    for i in range(n):
        d[i] = i
    return d


def benchmark(stmt, *, number=100, repeat_count=5):
    times = repeat(stmt, globals=globals(), number=number, repeat=repeat_count)
    return {
        "min": min(times),
        "median": median(times),
        "mean": mean(times),
        "all": times,
    }


construction_plain = benchmark(
    "create_dict(10_000)",
    number=100,
    repeat_count=5,
)

construction_ordered = benchmark(
    "create_ordered_dict(10_000)",
    number=100,
    repeat_count=5,
)

print("plain  :", construction_plain)
print("ordered:", construction_ordered)


plain  : {'min': 0.05901839956641197, 'median': 0.1151264002546668, 'mean': 0.103613300062716, 'all': [0.13900010008364916, 0.11934460047632456, 0.1151264002546668, 0.08557699993252754, 0.05901839956641197]}
ordered: {'min': 0.1260249000042677, 'median': 0.19614600017666817, 'mean': 0.19537122026085854, 'all': [0.1262924000620842, 0.20806120056658983, 0.19614600017666817, 0.1260249000042677, 0.3203316004946828]}


## Problem 22 — Benchmark lookup fairly

Create both mappings once, then benchmark repeated lookup of the same existing key.


In [32]:
# Solution 22

plain_lookup = create_dict(100_000)
ordered_lookup = create_ordered_dict(100_000)

plain_time = timeit(
    "plain_lookup[99_999]",
    globals=globals(),
    number=1_000_000,
)

ordered_time = timeit(
    "ordered_lookup[99_999]",
    globals=globals(),
    number=1_000_000,
)

print("plain lookup  :", plain_time)
print("ordered lookup:", ordered_time)
print("ratio ordered/plain:", ordered_time / plain_time)


plain lookup  : 0.06683950033038855
ordered lookup: 0.05854509957134724
ratio ordered/plain: 0.8759057036925474


## Problem 23 — Benchmark destructive operations correctly

A destructive benchmark can be misleading if one dictionary is exhausted and then reused.

Write benchmark helpers that create fresh input for each measured run.


In [33]:
# Solution 23

def pop_all_last_plain(n):
    d = create_dict(n)
    while d:
        d.popitem()


def pop_all_last_ordered(n):
    d = create_ordered_dict(n)
    while d:
        d.popitem(last=True)


def pop_all_first_plain(n):
    d = create_dict(n)
    while d:
        popitem_plain(d, last=False)


def pop_all_first_ordered(n):
    d = create_ordered_dict(n)
    while d:
        d.popitem(last=False)


n = 20_000

tests = {
    "plain_pop_last": "pop_all_last_plain(n)",
    "ordered_pop_last": "pop_all_last_ordered(n)",
    "plain_pop_first": "pop_all_first_plain(n)",
    "ordered_pop_first": "pop_all_first_ordered(n)",
}

for label, stmt in tests.items():
    timings = repeat(stmt, globals=globals(), number=1, repeat=5)
    print(f"{label:20s} min={min(timings):.6f}s median={median(timings):.6f}s")


plain_pop_last       min=0.003373s median=0.003718s
ordered_pop_last     min=0.004399s median=0.005827s
plain_pop_first      min=0.113298s median=0.239806s
ordered_pop_first    min=0.008154s median=0.013501s


## Problem 24 — Benchmark move-to-front workloads

Compare:

- `OrderedDict.move_to_end(key, last=False)`
- the plain-dict rotation implementation

Use a dictionary large enough to expose scaling differences.


In [34]:
# Solution 24

def move_front_ordered_once(n):
    d = OrderedDict((i, i) for i in range(n))
    d.move_to_end(n // 2, last=False)
    return d


def move_front_plain_once(n):
    d = {i: i for i in range(n)}
    move_to_front_plain(d, n // 2)
    return d


for n in (100, 1_000, 10_000):
    ordered_t = min(repeat(
        "move_front_ordered_once(n)",
        globals=globals(),
        number=10,
        repeat=3,
    ))

    plain_t = min(repeat(
        "move_front_plain_once(n)",
        globals=globals(),
        number=10,
        repeat=3,
    ))

    print(
        f"n={n:>6} | OrderedDict={ordered_t:.6f}s "
        f"| plain dict={plain_t:.6f}s "
        f"| ratio={plain_t / ordered_t:.2f}x"
    )


n=   100 | OrderedDict=0.000277s | plain dict=0.000216s | ratio=0.78x
n=  1000 | OrderedDict=0.002960s | plain dict=0.002291s | ratio=0.77x
n= 10000 | OrderedDict=0.035406s | plain dict=0.021822s | ratio=0.62x


## Problem 25 — Approximate shallow memory footprint

Use `sys.getsizeof` to compare container-level size for several mapping sizes.

Important: `sys.getsizeof` reports only the shallow size of the container object, not a recursive total of all keys/values.


In [35]:
# Solution 25

sizes = [0, 10, 100, 1_000, 10_000]

print(f"{'n':>8} {'dict bytes':>12} {'OrderedDict bytes':>18}")
print("-" * 42)

for n in sizes:
    d = {i: i for i in range(n)}
    od = OrderedDict((i, i) for i in range(n))
    print(f"{n:8d} {sys.getsizeof(d):12d} {sys.getsizeof(od):18d}")


       n   dict bytes  OrderedDict bytes
------------------------------------------
       0           64                128
      10          352                864
     100         4688              10000
    1000        36952              85400
   10000       294992             746128


# Part IX — Randomized Invariant Testing

## Problem 26 — Random operation differential test

Use random operations to verify that your plain-dict compatibility functions behave like `OrderedDict` for:

- move to end
- move to front
- pop first
- pop last

Run many operations and compare after every step.


In [36]:
# Solution 26

def randomized_differential_test(seed=0, steps=500):
    rng = random.Random(seed)

    initial = [(i, i * 10) for i in range(20)]
    plain = dict(initial)
    ordered = OrderedDict(initial)

    for step in range(steps):
        if not plain:
            items = [(i, i * 10) for i in range(20)]
            plain.update(items)
            ordered.update(items)

        action = rng.choice([
            "move_end",
            "move_front",
            "pop_first",
            "pop_last",
        ])

        key = rng.choice(list(plain))

        if action == "move_end":
            move_to_end_compatible(plain, key, last=True)
            ordered.move_to_end(key, last=True)

        elif action == "move_front":
            move_to_end_compatible(plain, key, last=False)
            ordered.move_to_end(key, last=False)

        elif action == "pop_first":
            got_plain = popitem_plain(plain, last=False)
            got_ordered = ordered.popitem(last=False)
            assert got_plain == got_ordered

        elif action == "pop_last":
            got_plain = popitem_plain(plain, last=True)
            got_ordered = ordered.popitem(last=True)
            assert got_plain == got_ordered

        assert_same_items_and_order(plain, ordered)

    return True


for seed in range(10):
    assert randomized_differential_test(seed=seed, steps=300)

print("Randomized differential tests passed.")


Randomized differential tests passed.


## Problem 27 — Property: moving a key preserves the key/value set

For random dictionaries and random target keys, verify that moving a key changes only order, never content.


In [37]:
# Solution 27

rng = random.Random(12345)

for _ in range(200):
    keys = list(range(rng.randint(1, 50)))
    rng.shuffle(keys)

    d = {k: k * k for k in keys}
    before = dict(d)

    target = rng.choice(list(d))
    move_to_end_compatible(d, target, last=rng.choice([True, False]))

    assert d == before
    assert set(d.items()) == set(before.items())

print("Content-preservation property passed.")


Content-preservation property passed.


# Part X — Design Problems

## Problem 28 — Ordered event log with promotion

Build an `EventLog` class backed by a plain `dict`.

Requirements:

- `record(event_id, payload)` inserts or updates.
- Updating an existing event should move it to the end.
- `promote(event_id)` moves it to the beginning.
- `pop_oldest()` removes the first event.
- `pop_newest()` removes the last event.
- `items()` exposes events in current order.


In [38]:
# Solution 28

class EventLog:
    def __init__(self):
        self._events = {}

    def record(self, event_id, payload):
        if event_id in self._events:
            self._events.pop(event_id)
        self._events[event_id] = payload

    def promote(self, event_id):
        move_to_front_plain(self._events, event_id)

    def pop_oldest(self):
        return popitem_plain(self._events, last=False)

    def pop_newest(self):
        return popitem_plain(self._events, last=True)

    def items(self):
        return self._events.items()

    def __len__(self):
        return len(self._events)


log = EventLog()
log.record("e1", {"kind": "login"})
log.record("e2", {"kind": "purchase"})
log.record("e3", {"kind": "logout"})

log.record("e1", {"kind": "login", "retry": True})
assert [k for k, _ in log.items()] == ["e2", "e3", "e1"]

log.promote("e3")
assert [k for k, _ in log.items()] == ["e3", "e2", "e1"]

assert log.pop_oldest()[0] == "e3"
assert log.pop_newest()[0] == "e1"

print(list(log.items()))


[('e2', {'kind': 'purchase'})]


## Problem 29 — Priority-by-touch scheduler

Use `OrderedDict` to implement a scheduler where:

- tasks are inserted at the end,
- touching a task makes it most recent,
- promoting a task makes it first,
- processing removes the first task.

This combines the exact operations emphasized by the source.


In [39]:
# Solution 29

class TaskScheduler:
    def __init__(self):
        self._tasks = OrderedDict()

    def add(self, task_id, payload):
        self._tasks[task_id] = payload

    def touch(self, task_id):
        self._tasks.move_to_end(task_id)

    def promote(self, task_id):
        self._tasks.move_to_end(task_id, last=False)

    def process_next(self):
        if not self._tasks:
            raise KeyError("no tasks")
        return self._tasks.popitem(last=False)

    def snapshot(self):
        return list(self._tasks.items())


scheduler = TaskScheduler()

scheduler.add("compile", 5)
scheduler.add("test", 10)
scheduler.add("deploy", 20)

scheduler.touch("compile")
assert [k for k, _ in scheduler.snapshot()] == ["test", "deploy", "compile"]

scheduler.promote("deploy")
assert [k for k, _ in scheduler.snapshot()] == ["deploy", "test", "compile"]

print("processing:", scheduler.process_next())
print("remaining :", scheduler.snapshot())


processing: ('deploy', 20)
remaining : [('test', 10), ('compile', 5)]


## Problem 30 — Detect order-only changes between configurations

Two configuration mappings can contain exactly the same data but differ in order.

Write:

```python
compare_configurations(old, new)
```

Return one of:

- `"identical"`
- `"order-only-change"`
- `"content-change"`


In [40]:
# Solution 30

def compare_configurations(old, new):
    if ordered_mapping_equal(old, new):
        return "identical"

    if old == new:
        return "order-only-change"

    return "content-change"


a = {"host": "localhost", "port": 8000, "debug": False}
b = {"port": 8000, "host": "localhost", "debug": False}
c = {"host": "localhost", "port": 9000, "debug": False}

assert compare_configurations(a, a.copy()) == "identical"
assert compare_configurations(a, b) == "order-only-change"
assert compare_configurations(a, c) == "content-change"

print(compare_configurations(a, b))


order-only-change


# Part XI — Debugging and Code Review

## Problem 31 — Find the bug

Consider:

```python
def move_front_buggy(d, key):
    d[key] = d.pop(key)
    for k in d:
        if k == key:
            break
        d[k] = d.pop(k)
```

Why is this unsafe?

### Solution

The loop iterates directly over `d` while changing dictionary structure/order with `pop` and reinsertion. Structural mutation during iteration can raise `RuntimeError` and is difficult to reason about.

The safe version snapshots the keys that will be rotated before mutating.


In [41]:
# Correct version revisited

def move_front_safe(d, key):
    d[key] = d.pop(key)
    keys_to_rotate = list(d)[:-1]

    for k in keys_to_rotate:
        d[k] = d.pop(k)


d = {"a": 1, "b": 2, "c": 3, "d": 4}
move_front_safe(d, "c")

assert list(d) == ["c", "a", "b", "d"]
print(d)


{'c': 3, 'a': 1, 'b': 2, 'd': 4}


## Problem 32 — Make a source-style equality function more concise

A loop-based implementation can first check mapping equality, then compare key order.

Rewrite it using `all(...)` and `zip(...)`, but keep it readable.


In [42]:
# Solution 32

def dict_equal_sensitive(left, right):
    return (
        left == right
        and all(k1 == k2 for k1, k2 in zip(left, right))
    )


d1 = {"a": 1, "b": 2, "c": 3}
d2 = {"a": 1, "b": 2, "c": 3}
d3 = {"b": 2, "a": 1, "c": 3}

assert dict_equal_sensitive(d1, d2)
assert not dict_equal_sensitive(d1, d3)

print("Tests passed.")


Tests passed.


# Part XII — Challenge Problems

## Problem 33 — Reorder according to a partial priority list

Given a mapping and a sequence of priority keys:

```python
priority = ["d", "b"]
```

Move those keys to the front **in exactly that order**, while preserving the relative order of all remaining keys.


In [43]:
# Solution 33

def prioritize_keys(d, priority):
    missing = [k for k in priority if k not in d]
    if missing:
        raise KeyError(f"missing priority keys: {missing}")

    priority_set = set(priority)
    reordered = [(k, d[k]) for k in priority]
    reordered.extend((k, v) for k, v in d.items() if k not in priority_set)

    d.clear()
    d.update(reordered)


d = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
prioritize_keys(d, ["d", "b"])

assert list(d) == ["d", "b", "a", "c", "e"]
print(d)


{'d': 4, 'b': 2, 'a': 1, 'c': 3, 'e': 5}


## Problem 34 — Merge mappings while controlling order

Write:

```python
merge_keep_first_position(left, right)
```

Rules:

- Start with `left` order.
- Values from `right` overwrite duplicate keys.
- Overwriting a duplicate key must **not** move its original position.
- New keys from `right` are appended in `right` order.


In [44]:
# Solution 34

def merge_keep_first_position(left, right):
    result = dict(left)
    result.update(right)
    return result


left = {"a": 1, "b": 2, "c": 3}
right = {"b": 200, "d": 4, "a": 100, "e": 5}

merged = merge_keep_first_position(left, right)

assert list(merged.items()) == [
    ("a", 100),
    ("b", 200),
    ("c", 3),
    ("d", 4),
    ("e", 5),
]

print(merged)


{'a': 100, 'b': 200, 'c': 3, 'd': 4, 'e': 5}


## Problem 35 — Merge while making overwritten keys most recent

Now change the semantics:

- Start from `left`.
- Process `right` from left to right.
- Every key touched by `right` should end up at the end in right-side processing order.


In [45]:
# Solution 35

def merge_touch_to_end(left, right):
    result = dict(left)

    for key, value in right.items():
        if key in result:
            result.pop(key)
        result[key] = value

    return result


left = {"a": 1, "b": 2, "c": 3}
right = {"b": 200, "d": 4, "a": 100, "e": 5}

merged = merge_touch_to_end(left, right)

assert list(merged.items()) == [
    ("c", 3),
    ("b", 200),
    ("d", 4),
    ("a", 100),
    ("e", 5),
]

print(merged)


{'c': 3, 'b': 200, 'd': 4, 'a': 100, 'e': 5}


## Problem 36 — Rolling "most recently changed" registry

Create a registry where every update makes the key most recent.

Add:

```python
oldest()
newest()
pop_oldest()
```

Use a plain `dict`.


In [46]:
# Solution 36

class ChangeRegistry:
    def __init__(self):
        self._data = {}

    def set(self, key, value):
        if key in self._data:
            self._data.pop(key)
        self._data[key] = value

    def oldest(self):
        return first_item(self._data)

    def newest(self):
        return last_item(self._data)

    def pop_oldest(self):
        return popitem_plain(self._data, last=False)

    def snapshot(self):
        return list(self._data.items())


registry = ChangeRegistry()
registry.set("alpha", 1)
registry.set("beta", 2)
registry.set("gamma", 3)
registry.set("alpha", 100)

assert registry.oldest() == ("beta", 2)
assert registry.newest() == ("alpha", 100)

print(registry.snapshot())


[('beta', 2), ('gamma', 3), ('alpha', 100)]


# Part XIII — Capstone

## Problem 37 — Adaptive recency map

Implement a reusable class that supports:

- ordinary lookup
- setting values
- optional "touch on get"
- move to front
- move to end
- pop oldest
- pop newest
- configurable capacity
- automatic eviction
- ordered snapshot
- membership and length

Use `OrderedDict` because the workload is explicitly reorder-heavy.


In [47]:
# Solution 37

class RecencyMap:
    def __init__(self, capacity=None, *, touch_on_get=False):
        if capacity is not None and capacity < 1:
            raise ValueError("capacity must be >= 1 or None")

        self.capacity = capacity
        self.touch_on_get = touch_on_get
        self._data = OrderedDict()

    def __contains__(self, key):
        return key in self._data

    def __len__(self):
        return len(self._data)

    def __getitem__(self, key):
        value = self._data[key]

        if self.touch_on_get:
            self._data.move_to_end(key)

        return value

    def __setitem__(self, key, value):
        if key in self._data:
            self._data[key] = value
            self._data.move_to_end(key)
        else:
            self._data[key] = value

        return self._evict_if_needed()

    def _evict_if_needed(self):
        if self.capacity is not None and len(self._data) > self.capacity:
            return self._data.popitem(last=False)
        return None

    def move_front(self, key):
        self._data.move_to_end(key, last=False)

    def move_end(self, key):
        self._data.move_to_end(key, last=True)

    def pop_oldest(self):
        return self._data.popitem(last=False)

    def pop_newest(self):
        return self._data.popitem(last=True)

    def snapshot(self):
        return list(self._data.items())

    def __repr__(self):
        return (
            f"RecencyMap(capacity={self.capacity}, "
            f"touch_on_get={self.touch_on_get}, "
            f"data={self.snapshot()})"
        )


In [48]:
# Capstone tests

m = RecencyMap(capacity=4, touch_on_get=True)

m["a"] = 1
m["b"] = 2
m["c"] = 3
m["d"] = 4

assert m.snapshot() == [
    ("a", 1),
    ("b", 2),
    ("c", 3),
    ("d", 4),
]

assert m["b"] == 2
assert [k for k, _ in m.snapshot()] == ["a", "c", "d", "b"]

evicted = m.__setitem__("e", 5)
assert evicted == ("a", 1)
assert [k for k, _ in m.snapshot()] == ["c", "d", "b", "e"]

m.move_front("e")
assert [k for k, _ in m.snapshot()] == ["e", "c", "d", "b"]

print(m)


RecencyMap(capacity=4, touch_on_get=True, data=[('e', 5), ('c', 3), ('d', 4), ('b', 2)])


# Part XIV — Final Review Questions with Answers

## Q1. If plain `dict` preserves insertion order, why might `OrderedDict` still be appropriate?

**Answer:** When the workload frequently reorders keys or explicitly needs operations such as moving a key to the front/end or popping from either end. `OrderedDict` communicates that order manipulation is part of the data structure's purpose.

## Q2. Does assigning a new value to an existing dictionary key move that key to the end?

**Answer:** No. To make an existing key newest in a plain `dict`, pop/delete it and insert it again.

## Q3. Why is `d1.keys() == d2.keys()` not an order-sensitive test?

**Answer:** Dictionary key views behave set-like for equality. Compare an ordered sequence instead, or iterate pairwise with `zip`.

## Q4. What is the main weakness of emulating move-to-front with a plain `dict`?

**Answer:** It generally requires linear work and often an auxiliary snapshot/rebuild, whereas `OrderedDict` is designed for efficient reordering.

## Q5. Why should destructive benchmarks recreate their input?

**Answer:** Otherwise later measurements may operate on a smaller or empty mapping, invalidating the comparison.

## Q6. When is plain `dict` the simpler choice?

**Answer:** When you primarily need ordinary mapping behavior plus insertion-order preservation, without heavy reordering operations.

## Q7. What is a useful testing strategy for compatibility helpers?

**Answer:** Differential testing: perform identical random operations on a plain-dict implementation and a reference `OrderedDict`, then compare ordered item sequences after every operation.


# Optional Extensions

Try these without looking up a ready-made solution:

1. Add `peek_oldest()` and `peek_newest()` to `RecencyMap`.
2. Add a `touch(key)` method that changes recency without changing the value.
3. Add `get(key, default)` semantics.
4. Add a callback invoked whenever capacity eviction happens.
5. Benchmark `move_to_front_plain` for `n = 10^2, 10^3, 10^4, 10^5`.
6. Write a CSV-style serializer whose output changes when key order changes.
7. Create a randomized test that mixes inserts, updates, deletes, moves, and pops.
8. Compare shallow memory sizes for integer keys vs long string keys.
9. Implement a plain-dict deque-like mapping API and document which operations are efficient.
10. Rewrite the LRU cache so that cache misses are counted separately from hits.


# Summary

The central design lesson is not simply that "`dict` is ordered now." The useful question is:

> **What ordered operations does my workload need, and how often?**

- If insertion order is enough, a plain `dict` is usually the simplest choice.
- If you need explicit, frequent reordering, `OrderedDict` provides clearer and more specialized operations.
- If you emulate those operations manually, test behavior, edge cases, complexity, and performance.
- For performance claims, benchmark the exact workload rather than relying on intuition.
